# Nemotron-Labs-Sigma-distill-spec-3B — self-speculative diffusion on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Nemotron-Labs-Sigma-distill-spec-3B-self-speculative-diffusion.ipynb)

[`nvidia/Nemotron-Labs-Sigma-distill-spec-3B`](https://huggingface.co/nvidia/Nemotron-Labs-Sigma-distill-spec-3B) is NVIDIA's **~3B continuous diffusion language model** trained for **self-speculative decoding**. One backbone carries two heads: the **diffusion head drafts a 32-token block in one call**, and the model's own **autoregressive head verifies** that draft. Accepted tokens are committed without another forward pass. The returned text is **exactly the greedy AR output** — acceptance only changes how many tokens each round commits, never which tokens.

It is a Nemotron-Labs-Sigma decoder (26 layers, hidden size 3072, FFN 9216, 32 query / 8 KV heads, 16-dimensional continuous token embeddings, degree-32 Bernstein noise schedule). Both heads ship in this repo, so the Hub lists about **4.24B stored parameters** (the extra AR readout is required for verification). Distilled with Phased DMD from an MS-SUSReg 3B teacher, then continued with an LK acceptance objective so the one-call draft is accepted more often. Self-conditioning is **off**. Native NFE is **2**; this speculative checkpoint uses **one** diffusion call per round (`diffusion_calls_per_round=1`).

**Why this model.** Ordinary speculative decoding needs a separate draft model. Here the same weights draft and verify, so you can measure **mean accepted length** (tokens kept per round, excluding the bonus token) on a free T4 without a second checkpoint. The [card](https://huggingface.co/nvidia/Nemotron-Labs-Sigma-distill-spec-3B) reports mean accepted length **10.49** on GSM8K CoT and a **3.28×** speedup over this model's own AR decode — those numbers are **SGLang, BF16, one NVIDIA GB200**, not this notebook.

**Use cases:** [when to pick this notebook](https://github.com/donvito/notebooks/blob/main/docs/Nemotron-Labs-Sigma-distill-spec-3B-self-speculative-diffusion.md)

**Licence:** [NVIDIA Nemotron Open Model License](https://www.nvidia.com/en-us/agreements/enterprise-software/nvidia-nemotron-open-model-license/). Not gated. `trust_remote_code=True` is required (`modeling_nemotron_labs_sigma.py`).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** of VRAM, **no native bfloat16**, and **no FlashAttention-2**. The card loads **bfloat16** and says **CPU is not supported**. Stored safetensors are about **8.5 GB** of BF16 (`total_size` 8,477,603,976 bytes). This notebook loads **float16** (~8.5 GB of weights). Full **float32** would be ~17 GB and **will not fit**. The remote sampler already runs the noise schedule in float64, the 16-dim embedding path and logits in float32, and RMSNorm / RoPE in float32; the encoder MLP / attention stay in the weight dtype. After load, a short probe checks that AR logits are finite. If they are not, the notebook falls back to **emulated bfloat16** (same 8.5 GB) and then to **float32 compute on the SwiGLU MLPs** (weights stay 2-byte). First run downloads ~8.5 GB. Attention is PyTorch **SDPA**.

`transformers>=4.55` as on the card. After the install cell, **restart the session** so Colab's already-imported transformers is replaced. Do **not** reinstall torch. Do **not** install `flash-attn`.

**Not in this notebook:** the pure few-step `model.generate()` block-diffusion sampler (that is the `-distill-*` sibling path, not AR-equivalent), SGLang serving, or the 8B distill-spec checkpoint.

**Links:** [model card](https://huggingface.co/nvidia/Nemotron-Labs-Sigma-distill-spec-3B) · [paper](https://arxiv.org/abs/2610.02665) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Nemotron-Labs-Sigma-distill-spec-3B-self-speculative-diffusion.md) · [source post](https://x.com/HuggingPapers/status/2122382659796021592)


## 0. Early GPU check

Print `torch.cuda.is_available()`, the GPU name, and free VRAM before installing anything. A restart after the install cell drops this state; section 2 prints it again and sets `DEVICE` / `DTYPE`.


In [ ]:
import shutil

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; this notebook will load float16.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install card deps (keep Colab's torch)

The card is PyTorch >= 2.4 and Hugging Face Transformers **>= 4.55**, loading this repo's inference code with `trust_remote_code`. Colab already has a CUDA torch. **Do not `pip install torch`.** Use `%pip`, not `python -I -m pip` (isolated mode drops Colab site-packages).

Do **not** install `flash-attn` — a T4 (`sm_75`) cannot run FlashAttention-2, and the Hub file uses `torch.nn.functional.scaled_dot_product_attention`.


In [ ]:
%pip install -q -U "transformers>=4.55" "jinja2>=3.1.6" accelerate safetensors huggingface_hub

# Card: PyTorch >= 2.4, transformers >= 4.55, trust_remote_code.
# jinja2>=3.1.6 is what transformers needs for apply_chat_template.
# Do NOT pip install torch. Colab already ships a CUDA wheel; forcing a
# reinstall often drops GPU support or pulls a CPU-only build.
# Do NOT install flash-attn (T4 / sm_75 cannot run FlashAttention 2).
# Use %pip, not python -I -m pip (isolated mode drops Colab site-packages).
print("install done — restart the runtime before the GPU-check cell")


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts. You do **not** need to rerun the install cell after the restart unless it failed.


## 2. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. A free T4 (~15 GB, `sm_75`) is the path this notebook is written for. The next cell stops if you are on CPU — the card does not support CPU inference.

T4 has **no native bfloat16**. Weights load in **float16**. Full float32 (~17 GB) does not fit.


In [ ]:
import gc
import inspect
import os
import sys
import time

import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("cuda available:", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("torch", torch.__version__, "| cuda", torch.version.cuda)

assert Version(transformers.__version__.split("+")[0]) >= Version("4.55"), (
    f"transformers {transformers.__version__} is loaded; the card needs "
    "transformers>=4.55. Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Hub code uses PyTorch SDPA.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
# Avoid a known source of fp16 overflow in large matmuls (vocab 131,072 heads).
if hasattr(torch.backends.cuda.matmul, "allow_fp16_reduced_precision_reduction"):
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False

DEVICE = "cuda"
MODEL_ID = "nvidia/Nemotron-Labs-Sigma-distill-spec-3B"
# Card default is bfloat16 (`config.json` torch_dtype, generation_config precision BF16).
# T4 has no native bf16 tensor cores. Full fp32 (~17 GB) does not fit ~15 GB.
DTYPE = torch.float16
# Card example uses max_new_tokens=256. Keep Run All short on a free T4.
MAX_NEW_TOKENS = 64

major, minor = torch.cuda.get_device_capability()
free_b, total_b = torch.cuda.mem_get_info()
props = torch.cuda.get_device_properties(0)

print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"dtype: {DTYPE}  (card default is bfloat16; T4 has no native bf16)")
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
print(f"device capability {major}.{minor} | multiprocessors {props.multi_processor_count}")
if major < 8:
    print("No native bfloat16 on this GPU — loading float16 weights (~8.5 GB).")
else:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")
print("FlashAttention-2: off (SDPA mem-efficient / math only)")
print(f"max_new_tokens for demos: {MAX_NEW_TOKENS}  (card snippet uses 256)")


def print_memory(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"{prefix}VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


## 3. Load Nemotron-Labs-Sigma-distill-spec-3B

`AutoTokenizer` + `AutoModel.from_pretrained(..., trust_remote_code=True)` — the Hub `auto_map` registers **`AutoModel`**, not `AutoModelForCausalLM`. First run downloads about **8.5 GB** (two BF16 shards). `device_map="cuda"` streams tensors onto the GPU so Colab's ~12.7 GB of host RAM is not asked to hold a full CPU copy.

After load, a prompt-sized forward through `_encode_clean` + `ar_head` checks that logits are finite. If they are not (fp16 overflow in the backbone), the cell tries **emulated bfloat16** (same 8.5 GB, slower on Turing) and then **float32 SwiGLU compute** with 2-byte weights. Full float32 weights are not attempted.


In [ ]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("bos:", tokenizer.bos_token, tokenizer.bos_token_id, "| eos:", tokenizer.eos_token, tokenizer.eos_token_id)
print("chat template starts with:", (tokenizer.chat_template or "")[:80].replace("\n", "\\n"))


def _from_pretrained(dtype):
    kw = dict(trust_remote_code=True, low_cpu_mem_usage=True)
    params = inspect.signature(AutoModel.from_pretrained).parameters
    if "dtype" in params:
        kw["dtype"] = dtype
    else:
        kw["torch_dtype"] = dtype
    t0 = time.perf_counter()
    try:
        mdl = AutoModel.from_pretrained(MODEL_ID, device_map={"": DEVICE}, **kw).eval()
    except (ValueError, ImportError, AttributeError) as err:
        print("device_map load failed, using .to(cuda):", type(err).__name__, err)
        mdl = AutoModel.from_pretrained(MODEL_ID, **kw).to(DEVICE).eval()
    torch.cuda.synchronize()
    print(f"loaded {MODEL_ID} dtype={dtype} in {time.perf_counter() - t0:.1f}s")
    return mdl


def _sigma_mod(mdl):
    return sys.modules[type(mdl).__module__]


def encode_user(text, tok=None):
    tok = tokenizer if tok is None else tok
    # Template already appends `<SPECIAL_11>Assistant\n` on the user turn, so
    # add_generation_prompt does not change the string (the jinja ignores it).
    ids = tok.apply_chat_template(
        [{"role": "user", "content": text}],
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    )
    if hasattr(ids, "input_ids"):
        ids = ids.input_ids
    elif isinstance(ids, list):
        ids = torch.tensor([ids], device=DEVICE)
        return ids
    return ids.to(DEVICE)


def ar_logits_are_finite(mdl, prompt="2 + 2 ="):
    # One causal prefix pass + AR head, as generate_speculative bootstraps.
    KVCache = _sigma_mod(mdl).KVCache
    input_ids = encode_user(prompt)
    b, length = input_ids.shape
    key_valid = torch.ones(b, length, dtype=torch.bool, device=DEVICE)
    pos = (key_valid.long().cumsum(dim=1) - 1).clamp_min(0)
    cache = KVCache(mdl.config.num_hidden_layers)
    with torch.inference_mode():
        hidden = mdl._encode_clean(input_ids, 0, pos, key_valid, cache)
        ar_logits = mdl.ar_head(hidden[:, -1]).float()
        diff_logits = mdl.diffusion_head(hidden[:, -1]).float()
    ar_ok = bool(torch.isfinite(ar_logits).all().item())
    diff_ok = bool(torch.isfinite(diff_logits).all().item())
    print(
        f"AR logits finite={ar_ok} min={ar_logits.min().item():.3g} max={ar_logits.max().item():.3g} | "
        f"diffusion logits finite={diff_ok} min={diff_logits.min().item():.3g} max={diff_logits.max().item():.3g}"
    )
    return ar_ok and diff_ok


def wrap_mlp_fp32_compute(mdl):
    # Keep 2-byte weights; upcast each SwiGLU matmul to float32, then cast back.
    # Casting the Linear modules themselves to fp32 would add ~4 GB and is a last resort.
    n = 0
    for layer in mdl.encoder.layers:
        mlp = layer.mlp
        if getattr(mlp, "_sigma_fp32_wrapped", False):
            continue
        gate_w, up_w, down_w = mlp.gate_proj.weight, mlp.up_proj.weight, mlp.down_proj.weight

        def fwd(x, _g=gate_w, _u=up_w, _d=down_w):
            h = x.float()
            y = torch.nn.functional.linear(
                torch.nn.functional.silu(torch.nn.functional.linear(h, _g.float()))
                * torch.nn.functional.linear(h, _u.float()),
                _d.float(),
            )
            return y.to(dtype=x.dtype)

        mlp.forward = fwd
        mlp._sigma_fp32_wrapped = True
        n += 1
    print(f"wrapped {n} encoder MLPs for float32 compute (weights unchanged)")


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model = _from_pretrained(DTYPE)

n_params = sum(p.numel() for p in model.parameters())
print("architecture:", getattr(model.config, "architectures", None))
print(
    "layers:", model.config.num_hidden_layers,
    "| hidden:", model.config.hidden_size,
    "| FFN:", model.config.intermediate_size,
    "| heads:", model.config.num_attention_heads, "/", model.config.num_key_value_heads,
)
print(
    "embed_dim:", model.config.embed_dim,
    "| block_size:", model.config.block_size,
    "| include_ar_head:", model.config.include_ar_head,
    "| native_nfe:", model.config.native_nfe,
)
print(
    "schedule:", model.config.schedule_parameterization,
    "| degree:", model.config.schedule_degree,
    "| selfcond config flag:", model.config.replaid_selfcond,
)
print(f"parameters: {n_params / 1e9:.3f}B | param dtype: {next(model.parameters()).dtype}")
print("has generate_speculative:", hasattr(model, "generate_speculative"))
print("has generate (block diffusion):", hasattr(model, "generate"))
print("has generate_ar:", hasattr(model, "generate_ar"))
print_memory("after load  ")

DTYPE_IN_USE = DTYPE
if not ar_logits_are_finite(model):
    print("Non-finite logits in float16 — retrying emulated bfloat16 (same 8.5 GB, slower on T4).")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    DTYPE_IN_USE = torch.bfloat16
    model = _from_pretrained(DTYPE_IN_USE)
    if not ar_logits_are_finite(model):
        print("Still non-finite — wrapping encoder MLPs for float32 compute, keeping 2-byte weights.")
        wrap_mlp_fp32_compute(model)
        if not ar_logits_are_finite(model):
            raise RuntimeError(
                "Logits are still non-finite after fp16, emulated bf16, and fp32 MLP compute. "
                "Full fp32 weights (~17 GB) will not fit a ~15 GB T4."
            )

print("Ready. DTYPE in use:", DTYPE_IN_USE)
print_memory("after probe  ")


## 4. Demo 1 — speculative generation

Two prompts, the card API:

```python
out_ids, stats = model.generate_speculative(
    input_ids,
    max_new_tokens=...,
    return_stats=True,        # rounds, accepted, mean_accepted_length
)
```

`draft_width` and `diffusion_calls_per_round` default to this checkpoint's trained contract (32 and 1; see `generation_config.json`). Drafts never use self-conditioning or guidance.

A short **math** prompt and a **general** writing prompt. Each cell prints the reply, mean accepted length, rounds, accepted draft tokens, committed tokens, wall time, and tokens/sec. `max_new_tokens` is `MAX_NEW_TOKENS` (64) so Run All stays interactive; the card snippet uses 256.


In [ ]:
PROMPTS = {
    "math": (
        "A shop starts with 70 apples. It sells 37 in the morning and 18 in the "
        "afternoon. How many apples are left? Write a short step-by-step solution."
    ),
    "general": (
        "In two short paragraphs, explain what a compiler does, using a kitchen "
        "recipe as an analogy."
    ),
}


@torch.no_grad()
def run_speculative(prompt, max_new_tokens=MAX_NEW_TOKENS, label=""):
    input_ids = encode_user(prompt)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    out_ids, stats = model.generate_speculative(
        input_ids,
        max_new_tokens=max_new_tokens,
        return_stats=True,
    )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    new_ids = out_ids[0, input_ids.shape[1]:]
    n_new = int(new_ids.numel())
    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    tps = n_new / elapsed if elapsed > 0 else float("nan")
    al = stats.get("mean_accepted_length", float("nan"))
    print("=" * 72)
    if label:
        print("LABEL:", label)
    print("PROMPT:", prompt)
    print()
    print("COMPLETION:")
    print(text)
    print()
    print(
        f"new tokens: {n_new} | rounds: {stats.get('rounds')} | "
        f"accepted (no bonus): {stats.get('accepted')} | committed: {stats.get('committed')}"
    )
    print(f"mean accepted length: {al:.2f}  (card: AL excludes the bonus token; a round commits AL+1)")
    print(f"{elapsed:.2f}s | {tps:.2f} tok/s")
    print_memory("after generate_speculative  ")
    return {
        "input_ids": input_ids,
        "out_ids": out_ids,
        "text": text,
        "stats": stats,
        "seconds": elapsed,
        "tok_s": tps,
        "n_new": n_new,
    }


spec_runs = {}
for key, prompt in PROMPTS.items():
    spec_runs[key] = run_speculative(prompt, label=key)
    print()


## 5. Demo 2 — match greedy AR on this GPU

`model.generate()` on this checkpoint is **block-diffusion sampling** (Phased DMD, `fresh_x0`, native NFE=2). That is a different decoder and is **not** the AR reference.

Remote code has **no** `generate_ar()`. The speculative docstring says the result is exactly greedy AR from `ar_head`. This cell therefore runs a token-by-token greedy loop on the same `_encode_clean` + `ar_head` path the verifier uses, then checks that the speculative tokens match.

Speedup is **measured on this T4** (do not substitute the card's GB200 / SGLang numbers). Speculative can be slower here: each round still runs a 32-token draft forward plus a verify forward, and T4 has no native BF16 and no FlashAttention-2.


In [ ]:
@torch.no_grad()
def greedy_ar_decode(mdl, input_ids, max_new_tokens=MAX_NEW_TOKENS):
    """Greedy AR using the same AR head generate_speculative verifies against."""
    KVCache = _sigma_mod(mdl).KVCache
    device = input_ids.device
    b = input_ids.shape[0]
    if b != 1:
        raise ValueError("greedy_ar_decode supports batch size 1, like generate_speculative.")
    key_valid = torch.ones(b, input_ids.shape[1], dtype=torch.bool, device=device)
    prompt_position_ids = (key_valid.long().cumsum(dim=1) - 1).clamp_min(0)
    prompt_lengths = key_valid.long().sum(dim=1)
    cache = KVCache(mdl.config.num_hidden_layers)
    hidden = mdl._encode_clean(input_ids, 0, prompt_position_ids, key_valid, cache)
    next_id = mdl.ar_head(hidden[:, -1]).argmax(dim=-1)
    clean = input_ids.clone()
    committed = 0
    while committed < max_new_tokens:
        clean = torch.cat([clean, next_id[:, None]], dim=1)
        committed += 1
        pos = prompt_lengths[:, None] + (committed - 1)
        key_valid = torch.cat(
            [key_valid, torch.ones(b, 1, dtype=torch.bool, device=device)], dim=1
        )
        hidden = mdl._encode_clean(next_id[:, None], clean.shape[1] - 1, pos, key_valid, cache)
        next_id = mdl.ar_head(hidden[:, -1]).argmax(dim=-1)
    return clean


COMPARE_KEY = "math"
compare_prompt = PROMPTS[COMPARE_KEY]
input_ids = encode_user(compare_prompt)

print("Comparing on the math prompt, max_new_tokens =", MAX_NEW_TOKENS)
print("PROMPT:", compare_prompt)
print()

torch.cuda.synchronize()
t0 = time.perf_counter()
ar_ids = greedy_ar_decode(model, input_ids, max_new_tokens=MAX_NEW_TOKENS)
torch.cuda.synchronize()
ar_seconds = time.perf_counter() - t0
ar_text = tokenizer.decode(ar_ids[0, input_ids.shape[1]:], skip_special_tokens=True)
ar_n = int(ar_ids.shape[1] - input_ids.shape[1])
ar_tps = ar_n / ar_seconds if ar_seconds > 0 else float("nan")

# Re-run speculative on the same input_ids so the match is not a cached coincidence.
torch.cuda.synchronize()
t1 = time.perf_counter()
spec_ids, spec_stats = model.generate_speculative(
    input_ids,
    max_new_tokens=MAX_NEW_TOKENS,
    return_stats=True,
)
torch.cuda.synchronize()
spec_seconds = time.perf_counter() - t1
spec_text = tokenizer.decode(spec_ids[0, input_ids.shape[1]:], skip_special_tokens=True)
spec_n = int(spec_ids.shape[1] - input_ids.shape[1])
spec_tps = spec_n / spec_seconds if spec_seconds > 0 else float("nan")

match = bool(torch.equal(ar_ids, spec_ids))
print("--- greedy AR (ar_head, token-by-token) ---")
print(ar_text)
print(f"{ar_seconds:.2f}s | {ar_n} tokens | {ar_tps:.2f} tok/s")
print()
print("--- generate_speculative ---")
print(spec_text)
print(
    f"{spec_seconds:.2f}s | {spec_n} tokens | {spec_tps:.2f} tok/s | "
    f"AL={spec_stats['mean_accepted_length']:.2f} | rounds={spec_stats['rounds']}"
)
print()
print("token ids match:", match)
if not match:
    n = min(ar_ids.shape[1], spec_ids.shape[1])
    same = (ar_ids[0, :n] == spec_ids[0, :n]).tolist()
    first_diff = next((i for i, ok in enumerate(same) if not ok), None)
    print("first differing index:", first_diff, "| AR len", ar_ids.shape[1], "| spec len", spec_ids.shape[1])
if spec_seconds > 0:
    print(f"measured T4 speedup (AR time / speculative time): {ar_seconds / spec_seconds:.2f}x")
print(
    "Card speedups (3.28x GSM8K, 2.61x HumanEval/MBPP, 1.98x SPEED-Bench) are "
    "SGLang BF16 on one NVIDIA GB200 — not this T4 SDPA path."
)
print_memory("after AR vs speculative  ")


## 6. How draft / verify works

One backbone, two heads, one round:

1. **Draft.** The diffusion head denoises a Gaussian sample in the 16-dimensional embedding space and reads out **32 tokens in one call** (`draft_width=32`, `diffusion_calls_per_round=1`). No self-conditioning, no guidance — matching how this checkpoint was trained and served.
2. **Verify.** The AR head scores that block in **one causal pass**. It keeps the longest prefix where draft token `j` equals the AR prediction for that position, then appends **one bonus token** of the AR head's own choosing.
3. **Commit.** A round therefore emits `AL + 1` tokens, where **AL** (`mean_accepted_length`) counts accepted **draft** tokens and **excludes** the bonus. The KV cache is trimmed to the verified prefix.

Because every kept token had to match greedy AR, the final string does not depend on how aggressive the drafter was — only the number of rounds does. That is why Demo 2 can check equality.

`model.generate()` is **not** this loop. It is few-step block diffusion (`native_nfe=2`, `sampler='fresh_x0'`). Use `generate_speculative` when you want AR-equivalent text plus acceptance stats.


In [ ]:
print("generation_config:", getattr(model, "generation_config", None))
print("config.block_size:", model.config.block_size)
print("config.native_nfe:", model.config.native_nfe, "| native_family:", model.config.native_family)
print("config.include_ar_head:", model.config.include_ar_head)
print()
print("Demo 1 stats (this T4 run, not the card's GB200 table):")
for key, run in spec_runs.items():
    s = run["stats"]
    print(
        f"  {key:8}  AL={s['mean_accepted_length']:.2f}  "
        f"rounds={s['rounds']}  accepted={s['accepted']}  "
        f"committed={s['committed']}  {run['tok_s']:.2f} tok/s"
    )


## 7. Peak VRAM

Allocated / reserved / peak after the demos. Weights are ~8.5 GB in 16-bit; the KV cache for 26 GQA layers at these short lengths is small. Full float32 weights (~17 GB) are not loaded.


In [ ]:
print_memory("session  ")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"dtype in use: {DTYPE_IN_USE} | parameters: {n_params / 1e9:.3f}B")
free_now, total_now = torch.cuda.mem_get_info()
print(f"free / total: {free_now / 1024**3:.2f} / {total_now / 1024**3:.2f} GB")


## 8. Cleanup

Release the checkpoint so later cells (or a second model) get the VRAM back.


In [ ]:
print_memory("before cleanup  ")
for name in ("model", "tokenizer", "spec_runs", "ar_ids", "spec_ids", "input_ids"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print_memory("after cleanup   ")
print("Released. Re-run the load cell if you want another generation.")


## Caveats

**Self-speculative, not a tool agent.** This notebook measures draft/verify on a T4. It is not a chat-with-tools loop. For a Nemotron tool agent on T4 see [Nemotron-3.5 Bolt E4B](nemotron-3.5-bolt-e4b-tool-agent.ipynb).

**`trust_remote_code`.** [`modeling_nemotron_labs_sigma.py`](https://huggingface.co/nvidia/Nemotron-Labs-Sigma-distill-spec-3B/blob/main/modeling_nemotron_labs_sigma.py) and [`configuration_nemotron_labs_sigma.py`](https://huggingface.co/nvidia/Nemotron-Labs-Sigma-distill-spec-3B/blob/main/configuration_nemotron_labs_sigma.py) run locally. Review them, and pin a revision for reproduction.

**API.** Load with `AutoModel` (not `AutoModelForCausalLM`). Generation that matches greedy AR is `model.generate_speculative(...)`. `model.generate(...)` is Phased-DMD block diffusion and is not AR-equivalent. There is no `generate_ar()`; Demo 2 uses the public `ar_head` / `_encode_clean` used by the verifier.

**T4 / dtype.** Card: BF16 on NVIDIA GPU, CPU not supported. This notebook: float16 weights, SDPA, no FlashAttention-2. Noise schedule stays float64; embedding path, logits, RMSNorm, and RoPE already run in float32 in the Hub file. If the probe sees NaNs, emulated bfloat16 then fp32 MLP compute. Full fp32 weights (~17 GB) will not fit. Card quality and speedup numbers used BF16 on GB200 with SGLang.

**VRAM.** Download ~8.5 GB. 4.24B parameters × 2 bytes ≈ 8.5 GB in VRAM after load, plus a small KV cache. Peak during 64-token demos should stay in the ~9–11 GB range on 15 GB — an estimate from the weight size, **not a measured Colab log attached to this file**. The notebook prints the real peak when you run it.

**Acceptance stats.** `mean_accepted_length` excludes the bonus token (card / paper AL). A round commits AL+1 tokens. The AR bootstrap round is excluded from the card's benchmark table; this notebook reports whatever `generate_speculative(..., return_stats=True)` returns.

**Length.** Context is 4096 tokens total. Demos stay at `MAX_NEW_TOKENS=64` so Run All finishes on a free T4. Raise it toward the card's 256 if you want longer samples.

**Related.** [ARLoop-1.2B](ARLoop-1.2B-looped-transformer.ipynb) is another `custom_code` T4 LM. [LFM3-1.7B chat and tools](LFM3-1.7B-chat-tools.ipynb) is a chat-tuned assistant, not a speculative decoder.
